# Batch Block Synchronization

Run the synchronization pipeline on **multiple blocks** in sequence with a clear folder layout and manual ROI selection:

1. **Named analysis subfolder** — You choose a name (default: `batch_analysis_output_yyyy_mm_dd`). Each block writes to `block_path/analysis/<name>/` so existing files are untouched.
2. **ROI then brightness** — First, define ROIs manually for every block (one pass). Then compute brightness vectors for all blocks using those ROIs.
3. **Automatic sync** — After brightness is ready, run full sync (LED alignment + drift correction) for all blocks and export `final_sync_df`.
4. **Report** — A report folder in the experiment directory holds a processing log (corrections, etc.) for verification.
5. **Channel mapping** — If `channeldict_by_animal` has no entry for the animal, the manual TTL line selector runs on an example block and a new key is added (and logged).

## 1. Imports

In [ ]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime
import sys
import json
import numpy as np
import pandas as pd
import cv2

from bokeh.io import output_notebook, show
from bokeh.models import Tabs, TabPanel
output_notebook()

from eye_tracking_system_tools.preprocessing import utility_functions as uf
from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import Manager
import threading

try:
    from tqdm.notebook import tqdm as tqdm_notebook
except ImportError:
    from tqdm import tqdm as tqdm_notebook

from eye_tracking_system_tools.preprocessing.block_sync_core import (
    simple_sync_build,
    shift_eye_df_by_index,
    compute_led_alignment_shift,
    apply_drift_correction,
    build_final_sync_df_merge_nearest,
    compute_sync_self_verification_correlation,
    run_jitter_report_worker,
)
from eye_tracking_system_tools.preprocessing.block_sync_visualization import (
    plot_sync_verification_verbose_bokeh,
)

## 2. Configuration

Set experiment path, block numbers, animal, and (optionally) channel mapping. **Analysis subfolder name** is where all outputs for this run will go; default is `batch_analysis_output_yyyy_mm_dd`.

In [ ]:
experiment_path = Path(r"D:\sample_data_for_eye_repo")
block_numbers = [6, 7]
animal = "PV_126"
bad_blocks = []

# Name for the analysis subfolder (same name used for every block). Default: batch_analysis_output_yyyy_mm_dd
analysis_subfolder_name = "batch_analysis_output_" + datetime.now().strftime("%Y_%m_%d")
# Or set explicitly, e.g.: analysis_subfolder_name = "my_run_20250204"

# Optional: continue from a previous run by renaming that analysis folder to today's name.
# Set to the previous folder name (e.g. "batch_analysis_output_2025_02_05") to rename it to analysis_subfolder_name
# for all blocks that have it. Leave None to start fresh.
previous_analysis_subfolder_name = None  # e.g. "batch_analysis_output_2025_02_05"

# Channel mapping (line -> role). If animal is missing, manual TTL selector runs on first block and is logged.
channeldict_by_animal = {
    "PV_208": {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"},
    "TE_21": {1: "Arena_TTL", 4: "LED_driver", 5: "R_eye_TTL", 8: "L_eye_TTL"},
    "PV_106": {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"},
    "PV_126": {1: "LED_driver", 7: "L_eye_TTL", 2: "Arena_TTL", 8: "R_eye_TTL"},
}
channeldict = channeldict_by_animal.get(animal)

## 3. Setup blocks and run folders

Build block list and set `analysis_path` to `analysis/<analysis_subfolder_name>` for each block. If **previous_analysis_subfolder_name** is set in Configuration, any block that has that analysis folder will have it renamed to today's name first, so you can continue where you left off. If **channeldict** is missing for this animal, the manual TTL line selector runs on the first block; the mapping is stored in `channeldict_by_animal` and all blocks use it (and a log entry is written later).

In [ ]:
def create_run_folder_for_block(block, analysis_subfolder_name):
    """Set block.analysis_path to block_path/analysis/<analysis_subfolder_name>."""
    base = block.block_path / "analysis"
    base.mkdir(parents=True, exist_ok=True)
    run_path = base / analysis_subfolder_name
    run_path.mkdir(parents=True, exist_ok=True)
    block.analysis_path = run_path
    return run_path


block_collection = list(uf.block_generator(
    block_numbers=block_numbers,
    experiment_path=experiment_path,
    animal=animal,
    bad_blocks=bad_blocks,
))

manual_ttl_used = False
if channeldict is None:
    # Run manual TTL selector via parse_open_ephys_events on first block; then read sidecar and use for all blocks.
    manual_ttl_used = True
    first_block = block_collection[0]
    first_block.channeldict = None
    first_block.handle_eye_videos()
    first_block.parse_open_ephys_events(overwrite=True, interactive_on_fail=True)
    sidecar = first_block.block_path / "oe_files" / first_block.oe_dirname / "ttl_manual_mapping.json"
    if sidecar.exists():
        data = json.loads(sidecar.read_text())
        channeldict = {int(line): name for name, line in data["manual_line_map"].items()}
        channeldict_by_animal[animal] = channeldict
        print(f"[Manual TTL] Created channeldict for {animal}: {channeldict}")
    else:
        raise RuntimeError("Manual TTL ran but ttl_manual_mapping.json not found.")

# Optional: rename a previous dated analysis folder to today so you can continue where you left off.
if previous_analysis_subfolder_name and previous_analysis_subfolder_name.strip():
    prev = previous_analysis_subfolder_name.strip()
    renamed = 0
    for b in block_collection:
        old_path = b.block_path / "analysis" / prev
        new_path = b.block_path / "analysis" / analysis_subfolder_name
        if not old_path.exists():
            continue
        if new_path.exists() and old_path.resolve() != new_path.resolve():
            print(f"[Block {b.block_num}] Skip rename: {analysis_subfolder_name} already exists.")
            continue
        if old_path.resolve() == new_path.resolve():
            continue
        old_path.rename(new_path)
        renamed += 1
        print(f"[Block {b.block_num}] Renamed analysis folder {prev} -> {analysis_subfolder_name}")
    if renamed:
        print(f"Renamed {renamed} block(s) to today's analysis folder.")

for b in block_collection:
    b.channeldict = channeldict
    create_run_folder_for_block(b, analysis_subfolder_name)

print(f"Blocks: {[b.block_num for b in block_collection]}")
print(f"Analysis subfolder: {analysis_subfolder_name}")
print(f"Example run path: {block_collection[0].analysis_path}")

## 4. Report folder

Create a report folder in the experiment directory. The sync step will write a processing log here.

In [ ]:
report_folder = experiment_path / "batch_sync_report"
report_folder.mkdir(parents=True, exist_ok=True)
report_log_path = report_folder / f"sync_log_{analysis_subfolder_name}.txt"
print(f"Report log will be written to: {report_log_path}")

## 5. Collect ROIs (manual) for all blocks

For each block, you will be asked to select an ROI for the **left** and **right** eye (first frame of each video). ROIs are stored; brightness is computed in the next step. Close each ROI window after drawing the rectangle.

In [ ]:
rois_by_block = {}

for block in block_collection:
    block.handle_eye_videos()
    if not block.le_videos or not block.re_videos:
        raise FileNotFoundError(f"Block {block.block_num}: no eye videos found.")
    rois = {}
    for eye_label, vid_path in [("Left Eye", block.le_videos[0]), ("Right Eye", block.re_videos[0])]:
        cap = cv2.VideoCapture(str(vid_path))
        if not cap.isOpened():
            raise RuntimeError(f"Cannot open {vid_path}")
        ret, frame = cap.read()
        cap.release()
        if not ret:
            raise RuntimeError(f"Cannot read first frame: {vid_path}")
        roi = cv2.selectROI(f"Block {block.block_num} - {eye_label}", frame, showCrosshair=True, fromCenter=False)
        cv2.destroyWindow(f"Block {block.block_num} - {eye_label}")
        rois[eye_label] = tuple(map(int, roi))
    rois_by_block[block.block_num] = rois
    print(f"Block {block.block_num}: ROIs set.")

print("All ROIs collected. Run the next cell to compute brightness vectors.")

## 6. Compute brightness vectors

Using the ROIs from the previous step, compute brightness per eye for each block and save to each block's analysis run folder.

In [ ]:
import pickle
threshold_value = 30

for block in block_collection:
    rois = rois_by_block[block.block_num]
    block.le_frame_val_list = BlockSync.produce_frame_val_list_with_roi(
        block.le_videos[0], rois["Left Eye"], threshold_value
    )
    block.re_frame_val_list = BlockSync.produce_frame_val_list_with_roi(
        block.re_videos[0], rois["Right Eye"], threshold_value
    )
    p = block.analysis_path / "eye_brightness_values_dict.pkl"
    with open(p, "wb") as f:
        pickle.dump({"left_eye": block.le_frame_val_list, "right_eye": block.re_frame_val_list}, f)
    print(f"Block {block.block_num}: brightness saved to {p}")

print("Brightness vectors ready. Run the next cell to perform synchronization.")

## 7. Run synchronization for all blocks

For each block: parse OE events, handle arena, load brightness from run folder, then run simple_sync → LED alignment → drift correction → merge and export. Console output is also written to the report log (and a note if manual TTL was used).

In [ ]:
class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, data):
        for f in self.files:
            f.write(data)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()


def run_sync_for_block(block, verbose=True, run_self_verification=True):
    """Run sync for one block (brightness must already exist in block.analysis_path)."""
    block.parse_open_ephys_events()
    block.handle_arena_files()
    block.get_eye_brightness_vectors(threshold_value=30, export=False, use_auto_roi=False)

    dfL, dfR = simple_sync_build(block, export=True)
    block.find_led_blink_frames(plot=False)
    shift_left, shift_right = compute_led_alignment_shift(block, dfL, dfR)
    if verbose:
        print(f"[Block {block.block_num}] Shifts: left={shift_left}, right={shift_right}")
    dfL_shifted = shift_eye_df_by_index(dfL, shift_left)
    dfR_shifted = shift_eye_df_by_index(dfR, shift_right)
    dfL_corrected, dfR_corrected = apply_drift_correction(block, dfL_shifted, dfR_shifted, verbose=verbose)
    final_df = build_final_sync_df_merge_nearest(
        block, dfL_corrected, dfR_corrected,
        pre_shift_left=0, pre_shift_right=0,
        export_csv=True, csv_name="blocksync_df.csv", verbose=verbose,
    )
    block.final_sync_df = final_df
    dfL_corrected.sort_index().to_csv(block.analysis_path / "eye_left_corrected_sync.csv")
    dfR_corrected.sort_index().to_csv(block.analysis_path / "eye_right_corrected_sync.csv")

    if run_self_verification:
        verif = compute_sync_self_verification_correlation(
            block, dfL_corrected, dfR_corrected,
            window_seconds=0.5, min_correlation_threshold=0.4,
        )
        block.sync_self_verification = verif
        if verbose:
            status = "PASS" if verif["passed"] else "LOW_CORR"
            print(f"[Block {block.block_num}] Self-verification (L/R correlation at LED blinks): {status} (mean r={verif['mean_correlation']:.3f}, threshold={verif['min_correlation_threshold']})")
    return block, dfL_corrected, dfR_corrected


sync_results = []
sync_failures = []
with open(report_log_path, "w") as report_file:
    report_file.write(f"Batch sync report: {analysis_subfolder_name}\n")
    report_file.write(f"Experiment: {experiment_path}\n")
    report_file.write(f"Animal: {animal}  Blocks: {block_numbers}\n")
    if manual_ttl_used:
        report_file.write("\n[LOG] Manual TTL line selector was used for this animal; channeldict was created and applied to all blocks.\n")
    report_file.write("\n--- Processing log ---\n\n")
    report_file.flush()

    old_stdout = sys.stdout
    sys.stdout = Tee(old_stdout, report_file)
    try:
        for block in block_collection:
            print(f"\n[Block {block.block_num}] Run folder: {block.analysis_path}")
            try:
                block, dfL, dfR = run_sync_for_block(block, verbose=True)
                sync_results.append((block, dfL, dfR))
            except Exception as e:
                block.sync_status = "failed_to_analyze"
                block.sync_error = str(e)
                sync_failures.append((block, e))
                print(f"\n*** [Block {block.block_num}] FAILED TO ANALYZE ***")
                print(f"    Error: {e}")
                print(f"    Block marked as sync_status='failed_to_analyze'. Fix pre-analysis (e.g. missing/broken eye video) or run single-block processing, then retry.")
                report_file.write(f"\n[FAILED] Block {block.block_num}: {e}\n")
                report_file.flush()
                continue
        print(f"\nDone. Synced {len(sync_results)} block(s).")
        for block, _, _ in sync_results:
            print(f"  Block {block.block_num}: {block.analysis_path}")
        if sync_failures:
            print(f"\n*** {len(sync_failures)} block(s) failed_to_analyze (see above). Check report log and fix or process those blocks separately.")
            for block, err in sync_failures:
                print(f"  Block {block.block_num}: {err}")
    finally:
        sys.stdout = old_stdout

print(f"Report log written to: {report_log_path}")

## 8. Batch sync summary table

Table of each block's synchronization status, self-verification correlation (L/R brightness at LED blinks), and any error for failed blocks.

In [ ]:
# Build a table-style report for all blocks (synced + failed)
if "sync_results" not in dir():
    sync_results = []
if "sync_failures" not in dir():
    sync_failures = []
synced_blocks = {b.block_num: b for b, _, _ in sync_results}
failed_blocks = {b.block_num: (b, e) for b, e in sync_failures}

rows = []
for block in block_collection:
    bn = block.block_num
    if bn in synced_blocks:
        b = synced_blocks[bn]
        verif = getattr(b, "sync_self_verification", None)
        if verif is not None:
            mean_r = verif.get("mean_correlation", np.nan)
            passed = verif.get("passed", False)
            n_ev = verif.get("n_events", "")
            verif_status = "PASS" if passed else "LOW_CORR"
        else:
            mean_r = np.nan
            verif_status = "—"
            n_ev = ""
        rows.append({
            "Block": bn,
            "Status": "synced",
            "Mean correlation": mean_r if np.isfinite(mean_r) else "—",
            "Self-verification": verif_status,
            "N LED events": n_ev,
            "Error": "",
            "Analysis path": str(block.analysis_path) if getattr(block, "analysis_path", None) else "",
        })
    elif bn in failed_blocks:
        b, err = failed_blocks[bn]
        err_str = str(err)
        if len(err_str) > 80:
            err_str = err_str[:77] + "..."
        rows.append({
            "Block": bn,
            "Status": "failed_to_analyze",
            "Mean correlation": "—",
            "Self-verification": "—",
            "N LED events": "",
            "Error": err_str,
            "Analysis path": str(block.analysis_path) if getattr(block, "analysis_path", None) else "",
        })
    else:
        rows.append({
            "Block": bn,
            "Status": "not_processed",
            "Mean correlation": "—",
            "Self-verification": "—",
            "N LED events": "",
            "Error": "",
            "Analysis path": str(block.analysis_path) if getattr(block, "analysis_path", None) else "",
        })

report_df = pd.DataFrame(rows)
display(report_df)

## 8.1 Batch jitter reports

Jitter reports estimate frame-to-frame displacement (cross-correlation) per eye video. They require an ROI per eye (same idea as brightness: a region to correlate). You can **reuse the ROIs from step 5** (brightness) or define separate jitter ROIs below. ROIs for correlation must have odd width and height; if reusing brightness ROIs, they are adjusted automatically.

**Parallel execution:** The next cell runs one process per block (using multiple cores). Blocks that already have a saved `jitter_report_dict.pkl` are **not recomputed** unless you set `overwrite_jitter_reports = True`.

**Progress:** Set `jitter_verbose = True` (default) to show one tqdm progress bar per block, updated live by the workers so you can see how much each block has left.

In [ ]:
# Option: reuse ROIs from brightness (step 5). Set to False to define jitter ROIs manually.
use_brightness_rois_for_jitter = True


def _ensure_odd_roi(roi):
    """Ensure ROI has odd width and height (required for cross-correlation)."""
    x, y, w, h = roi
    if w % 2 == 0:
        w += 1
    if h % 2 == 0:
        h += 1
    return [x, y, w, h]


if use_brightness_rois_for_jitter and "rois_by_block" in dir() and rois_by_block:
    jitter_rois_by_block = {}
    for bn, rois in rois_by_block.items():
        jitter_rois_by_block[bn] = {
            "left_roi": _ensure_odd_roi(rois["Left Eye"]),
            "right_roi": _ensure_odd_roi(rois["Right Eye"]),
        }
    print(f"Reused brightness ROIs for jitter (odd-sized) for {len(jitter_rois_by_block)} block(s).")
else:
    jitter_rois_by_block = {}
    for block in block_collection:
        block.handle_eye_videos()
        if not block.le_videos or not block.re_videos:
            print(f"[Block {block.block_num}] Skipping: no eye videos.")
            continue
        rois = {}
        for eye_label, vid_path in [("Left eye (jitter)", block.le_videos[0]), ("Right eye (jitter)", block.re_videos[0])]:
            cap = cv2.VideoCapture(str(vid_path))
            if not cap.isOpened():
                raise RuntimeError(f"Cannot open {vid_path}")
            ret, frame = cap.read()
            cap.release()
            if not ret:
                raise RuntimeError(f"Cannot read first frame: {vid_path}")
            roi = list(cv2.selectROI(f"Block {block.block_num} - {eye_label}", frame, showCrosshair=True, fromCenter=False))
            cv2.destroyWindow(f"Block {block.block_num} - {eye_label}")
            roi = _ensure_odd_roi(roi)
            key = "left_roi" if "Left" in eye_label else "right_roi"
            rois[key] = roi
        jitter_rois_by_block[block.block_num] = rois
        print(f"Block {block.block_num}: jitter ROIs set.")
    print(f"Jitter ROIs collected for {len(jitter_rois_by_block)} block(s).")

In [ ]:
# Run jitter reports in parallel (one process per block). Existing reports are loaded, not overwritten,
# unless overwrite_jitter_reports=True. Set max_workers to cap parallelism (default: one per block).
overwrite_jitter_reports = False  # Set True to recompute and overwrite existing jitter_report_dict.pkl
max_workers = None  # None = one process per block; set e.g. 4 to limit parallel jobs
jitter_verbose = True  # Show one tqdm progress bar per block (updated live by workers)

blocks_with_rois = [b for b in block_collection if b.block_num in jitter_rois_by_block]
if not blocks_with_rois:
    print("No blocks have jitter ROIs. Run the ROI cell above first.")
else:
    worker_args = []
    for block in blocks_with_rois:
        block.handle_eye_videos()
        if not block.le_videos or not block.re_videos:
            print(f"[Block {block.block_num}] Skipping: no eye videos.")
            continue
        bn = block.block_num
        roi_dict = jitter_rois_by_block[bn]
        base_args = (
            str(block.le_videos[0]),
            str(block.re_videos[0]),
            roi_dict["left_roi"],
            roi_dict["right_roi"],
            str(block.analysis_path),
            overwrite_jitter_reports,
        )
        worker_args.append((base_args, bn))

    n_workers = min(len(worker_args), max_workers or len(worker_args))
    print(f"Running jitter reports for {len(worker_args)} block(s) using {n_workers} process(es). Existing reports loaded unless overwrite_jitter_reports=True.")
    jitter_results = []
    jitter_failures = []
    block_by_path = {str(b.analysis_path): b for b in blocks_with_rois}

    if jitter_verbose:
        manager = Manager()
        progress_queue = manager.Queue()
        full_args = [a[0] + (progress_queue, a[1]) for a in worker_args]
        bars = {}
        done_count = [0]
        num_blocks = len(full_args)

        def progress_listener():
            while done_count[0] < num_blocks:
                try:
                    msg = progress_queue.get(timeout=0.5)
                except Exception:
                    continue
                if msg[0] == "init":
                    _, block_id, total = msg
                    bars[block_id] = tqdm_notebook(total=total, desc=f"Block {block_id}", unit="frame")
                elif msg[0] == "progress":
                    _, block_id, current, total = msg
                    if block_id in bars:
                        bars[block_id].n = current
                        bars[block_id].refresh()
                elif msg[0] == "done":
                    _, block_id = msg
                    if block_id in bars:
                        bars[block_id].close()
                    done_count[0] += 1
            return None

        listener = threading.Thread(target=progress_listener, daemon=True)
        listener.start()
    else:
        full_args = [a[0] for a in worker_args]

    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(run_jitter_report_worker, a): a for a in full_args}
        for future in as_completed(futures):
            args = futures[future]
            ap = args[4]
            try:
                status, path_str, err = future.result()
                block = block_by_path.get(path_str)
                if status == "skipped":
                    print(f"  [Block {block.block_num}] Loaded existing jitter report.")
                elif status == "computed":
                    print(f"  [Block {block.block_num}] Jitter report computed and saved.")
                else:
                    jitter_failures.append((block, err or "unknown"))
                    print(f"  [Block {block.block_num}] FAILED: {err}")
                    continue
                if block is not None:
                    jitter_results.append(block)
            except Exception as e:
                block = block_by_path.get(ap, None)
                jitter_failures.append((block, str(e)))
                print(f"  Worker error ({ap}): {e}")

    if jitter_verbose and listener.is_alive():
        listener.join(timeout=5.0)

    for block in jitter_results:
        block.get_jitter_reports(overwrite=False, sort_on_loading=True)

    print(f"\nDone. Jitter reports: {len(jitter_results)} ok, {len(jitter_failures)} failed.")
    if jitter_failures:
        for block, err in jitter_failures:
            print(f"  Block {block.block_num if block else '?'}: {err}")

## 9. Interactive verification

Use the tabs to switch between blocks and inspect the verbose sync verification plot.

In [ ]:
if not sync_results:
    print("No sync results. Run the sync cell above first.")
    if sync_failures:
        print(f"{len(sync_failures)} block(s) failed_to_analyze; fix those or run single-block processing.")
else:
    tabs_list = []
    for block, dfL, dfR in sync_results:
        layout = plot_sync_verification_verbose_bokeh(block, dfL, dfR, tolerance_seconds=0.017)
        tabs_list.append(TabPanel(child=layout, title=f"Block {block.block_num}"))
    if sync_failures:
        print(f"Note: {len(sync_failures)} block(s) failed_to_analyze (not shown in tabs).")
    show(Tabs(tabs=tabs_list))

## 10. (Optional) Verify from disk

To re-verify a block later: set `run_folder` and load the block and corrected sync CSVs.

In [ ]:
# run_folder = Path(r"D:\...\block_006\analysis\batch_analysis_output_2025_02_04")
# block_num_verify = "006"
# 
# blocks = list(uf.block_generator(
#     block_numbers=[int(block_num_verify)], experiment_path=experiment_path, animal=animal, bad_blocks=bad_blocks
# ))
# for b in blocks:
#     b.channeldict = channeldict_by_animal.get(animal)
# block = blocks[0]
# block.analysis_path = run_folder
# block.handle_eye_videos()
# block.parse_open_ephys_events()
# block.handle_arena_files()
# block.get_eye_brightness_vectors(threshold_value=30, export=False, use_auto_roi=False)
# block.find_led_blink_frames(plot=False)
# dfL = pd.read_csv(run_folder / "eye_left_corrected_sync.csv", index_col=0)
# dfR = pd.read_csv(run_folder / "eye_right_corrected_sync.csv", index_col=0)
# show(plot_sync_verification_verbose_bokeh(block, dfL, dfR, tolerance_seconds=0.017))